In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler # NEU
import joblib
from torch.utils.data import TensorDataset, DataLoader

import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# 1. Laden des Datensatzes
df = pd.read_csv('Datasets/Titanic_train.csv')

# 2. Vorverarbeitung
df = df.drop(['PassengerId', 'Name', 'Ticket', 'Cabin'], axis=1)
df['Age']      = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])

le_gender   = LabelEncoder()
le_embarked = LabelEncoder()
df['Gender']   = le_gender.fit_transform(df['Gender'])    # female=0, male=1
df['Embarked'] = le_embarked.fit_transform(df['Embarked'])  # C=0, Q=1, S=2

print(df.head())
print("Shape:", df.shape)

In [ ]:
# 3. Features (X) und Target (y) trennen
# Features: Pclass, Gender, Age, SibSp, Parch, Fare, Embarked  -> 7 Merkmale
X = df.drop('Survived', axis=1).values
y = df['Survived'].values

# 4. Aufteilen des Datensatzes (Train / Val / Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=0)

# Feature Scaling ---
# Für neuronale Netze ist Skalierung essenziell für die Konvergenz. Gängige Methoden:
# 1. StandardScaler (Standardisierung): Mittelwert = 0, Standardabweichung = 1 (Standard für NNs)
# 2. MinMaxScaler (Normalisierung): Skaliert Werte starr in einen Bereich (meist 0 bis 1)
# 3. RobustScaler: Nutzt Median und Quartile, sehr robust gegenüber Ausreißern (Outliers)
scaler = StandardScaler()

# Rohwerte sichern - für den Vorher/Nachher-Vergleich
X_train_raw = X_train.copy()

# WICHTIG (Vermeidung von Data Leakage): 'fit_transform' NUR auf Trainingsdaten anwenden!
# Der Scaler lernt hier den Mittelwert und die Varianz der Trainingsdaten.
X_train = scaler.fit_transform(X_train)

# Auf Validierungs- und Testdaten NUR 'transform' anwenden!
# Sie werden mit den Parametern skaliert, die aus den Trainingsdaten gelernt wurden.
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)
# -----------------------------------------

# Visualisierung: Trainingsdaten vor und nach der Skalierung
feature_names = df.drop('Survived', axis=1).columns.tolist()
df_raw    = pd.DataFrame(X_train_raw, columns=feature_names)
df_scaled = pd.DataFrame(X_train,     columns=feature_names)
df_raw['Skalierung']    = 'Vor Skalierung'
df_scaled['Skalierung'] = 'Nach Skalierung'
df_compare = pd.concat([df_raw, df_scaled], ignore_index=True)
df_compare = df_compare.melt(id_vars='Skalierung', var_name='Merkmal', value_name='Wert')

# Zwei separate Subplots mit je eigener Y-Achse
# (gemeinsame Achse wäre durch Fare-Ausreißer ~512 dominiert – skalierte Werte wären unsichtbar)
df_vor  = df_compare[df_compare["Skalierung"] == "Vor Skalierung"]
df_nach = df_compare[df_compare["Skalierung"] == "Nach Skalierung"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=df_vor,  x='Merkmal', y='Wert', ax=ax1, color='#f4a261')
ax1.set_title('Vor Skalierung  (Rohwerte)')
ax1.tick_params(axis="x", rotation=20)
ax1.set_xlabel('Merkmal')
ax1.set_ylabel('Wert')

sns.boxplot(data=df_nach, x='Merkmal', y='Wert', ax=ax2, color='#457b9d')
ax2.axhline(0, color="gray", linewidth=0.8, linestyle="--")
ax2.set_title('Nach StandardScaler  (Mittelwert=0, σ=1)')
ax2.tick_params(axis="x", rotation=20)
ax2.set_xlabel('Merkmal')
ax2.set_ylabel('Wert')

fig.suptitle('Trainingsdaten: Vor vs. Nach StandardScaler', fontsize=13)
plt.tight_layout()
plt.show()

# 5. Umwandeln der skalierten Daten in PyTorch-Tensoren
X_train = torch.from_numpy(X_train).float()
X_test  = torch.from_numpy(X_test).float()
y_train = torch.from_numpy(y_train).long()
y_test  = torch.from_numpy(y_test).long()
X_val   = torch.from_numpy(X_val).float()
y_val   = torch.from_numpy(y_val).long()

# Erstellen eines TensorDatasets und DataLoaders - Kombinieren von Features und Labels zu einem Dataset
train_dataset = TensorDataset(X_train, y_train)
# Erstellen des DataLoaders für das Training
batch_size = 32
# shuffle=True sorgt dafür, dass die Daten in jeder Epoche gemischt werden
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

# Definieren des neuronalen Netzes als Sequential-Modell  (7 Features -> 16 Neuronen -> 2 Klassen)
net = nn.Sequential(
    nn.Linear(7, 16),
    nn.ReLU(),
    nn.Linear(16, 2)
)

# Definieren des Verlustkriteriums und des Optimierers
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(net.parameters(), lr=0.01)

loss_history = []

# Trainieren des neuronalen Netzes
net.train()  # Trainingsmodus aktivieren
for epoch in range(100):
    for batch_data in train_loader: # Schleife über die Batches aus dem DataLoader
        batch_X = batch_data[0]
        batch_y = batch_data[1]
        optimizer.zero_grad()
        outputs = net(batch_X)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()

    loss_history.append(loss.item())

    # Validierung nach jeder Epoche
    net.eval()
    with torch.no_grad():
        val_out  = net(X_val)
        val_loss = criterion(val_out, y_val)
    net.train()
    # das ist eigentlich der loss vom letzten batch. und nicht von allen zusammen!
    print(f'Epoch {epoch:3d} | Loss: {loss.item():.4f}')

# Loss-Kurve
plt.figure(figsize=(8, 4))
plt.plot(loss_history, label='Training Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Trainingsverlauf')
plt.legend()
plt.tight_layout()
plt.show()

# Auswerten des neuronalen Netzes auf den Testdaten
net.eval()
with torch.no_grad():
    outputs = net(X_test)
    _, predicted = torch.max(outputs, 1)
    accuracy = accuracy_score(y_test, predicted)
    print('Testgenauigkeit: ', accuracy)

# Abspeichern des trainierten Netzes
torch.save(net.state_dict(), 'Models/titanic_net_NextStep2.pth')
joblib.dump(le_gender,   'Models/le_gender_NextStep2.pkl')
joblib.dump(le_embarked, 'Models/le_embarked_NextStep2.pkl')
joblib.dump(scaler,      'Models/scaler_NextStep2.pkl')

# Confusion Matrix
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test.numpy(), predicted.numpy())
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Nicht überlebt', 'Überlebt'],
            yticklabels=['Nicht überlebt', 'Überlebt'])
plt.xlabel('Vorhergesagte Klasse')
plt.ylabel('Tatsächliche Klasse')
plt.title('Confusion Matrix: Titanic Survival')
plt.tight_layout()
plt.show()

In [ ]:
# Wiederladen des State-Dicts
import torch
import torch.nn as nn

net.load_state_dict(torch.load('Models/titanic_net_NextStep2.pth', map_location=torch.device('cpu')))
le_gender   = joblib.load('Models/le_gender_NextStep2.pkl')
le_embarked = joblib.load('Models/le_embarked_NextStep2.pkl')
scaler      = joblib.load('Models/scaler_NextStep2.pkl')

net = nn.Sequential(
    nn.Linear(7, 16),
    nn.ReLU(),
    nn.Linear(16, 2)
)
for k, v in net.named_parameters():
    print(k, v)
net.eval()

In [ ]:
# Vorhersage mit dem trainierten Netz

# Eingabe der Passagier-Merkmale
print("Geben Sie die Passagier-Merkmale ein:")
pclass   = int(input("Klasse (1 / 2 / 3): "))
gender   = input("Geschlecht (male / female): ").strip().lower()
age      = float(input("Alter: "))
sibsp    = int(input("Geschwister / Ehepartner an Bord (SibSp): "))
parch    = int(input("Eltern / Kinder an Bord (Parch): "))
fare     = float(input("Ticketpreis (Fare): "))
embarked = input("Einschiffungshafen (C / Q / S): ").strip().upper()

# Skalierung der Eingabewerte
import numpy as np
gender_enc   = le_gender.transform([gender])[0]
embarked_enc = le_embarked.transform([embarked])[0]
eingabe = np.array([[pclass, gender_enc, age, sibsp, parch, fare, embarked_enc]])
eingabe_skaliert = scaler.transform(eingabe)

# Erstellen eines Tensors aus den Eingabewerten
inputs = torch.tensor(eingabe_skaliert, dtype=torch.float32)

# Vorhersage treffen
with torch.no_grad():
    outputs = net(inputs)
    _, predicted = torch.max(outputs, 1)

# Ausgabe der Klassifizierungsaussage
result = "Überlebt" if predicted.item() == 1 else "Nicht überlebt"
print(f"Vorhersage: {result}")

In [ ]:
# aus dem obigen Skript wird inputs übernommen,
# die Vorhersagen werden als Wahrscheinlichkeiten ausgegeben

import torch.nn.functional as F

# Vorhersage des Netzes
with torch.no_grad():
    output = net(inputs)
    _, max_index = torch.max(output, 1)
    print("Vorhergesagte Klasse:", "Überlebt" if max_index.item() == 1 else "Nicht überlebt")

# Klassenwahrscheinlichkeiten berechnen
probabilities = F.softmax(output, dim=1)
print(f"Wahrscheinlichkeit Nicht überlebt: {probabilities[0][0]:.4f}")
print(f"Wahrscheinlichkeit Überlebt:       {probabilities[0][1]:.4f}")